# Attendance Prediction Models Comparison - PortAventura World

This notebook compares multiple forecasting models to predict monthly attendance for **PortAventura World**.
Given the small dataset size (approx 40 observations), we focus on robust models suitable for short time series with seasonality.

**Models Evaluated:**
1.  **Seasonal Naive**: Baseline
2.  **SARIMAX**: Classical statistical model with exogenous variables (for COVID)
3.  **ETS (Exponential Smoothing)**: Holt-Winters method
4.  **Prophet**: Robust to outliers and missing data
5.  **Theta**: Simple and often effective benchmark
6.  **Ensemble**: Average of top models

**Key Handling:**
-   **Yearly seasonality** (period=12)
-   **COVID period** (Feb 2020 - June 2021) impact analysis

In [ ]:
# Install necessary libraries if not present
!pip install prophet statsmodels scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.forecasting.theta import ThetaModel
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 6)
plt.rcParams['font.size'] = 10

## 1. Data Loading and Preprocessing

In [ ]:
# Load data
df = pd.read_csv('final_data/attendance.csv')
df['USAGE_DATE'] = pd.to_datetime(df['USAGE_DATE'])
portaventura = df[df['FACILITY_NAME'] == 'PortAventura World'].copy()

# Aggregate to Monthly
portaventura['year_month'] = portaventura['USAGE_DATE'].dt.to_period('M')
monthly_data = portaventura.groupby('year_month')['attendance'].sum().reset_index()
monthly_data['date'] = monthly_data['year_month'].dt.to_timestamp()
monthly_data = monthly_data.set_index('date').sort_index()

# Ensure continuous monthly frequency and fill missing values (e.g. park closures) with 0
monthly_data = monthly_data.asfreq('MS').fillna(0)

# Define COVID period
covid_start = '2020-03-01'
covid_end = '2021-06-01'

# Create exogenous variable for COVID (binary)
monthly_data['covid_dummy'] = 0
monthly_data.loc[(monthly_data.index >= covid_start) & (monthly_data.index <= covid_end), 'covid_dummy'] = 1

# Visualize
plt.figure(figsize=(15, 6))
plt.plot(monthly_data.index, monthly_data['attendance'], marker='o', label='Actual Attendance')
plt.axvspan(pd.to_datetime(covid_start), pd.to_datetime(covid_end), color='red', alpha=0.1, label='COVID Period')
plt.title('PortAventura Monthly Attendance')
plt.legend()
plt.show()

monthly_data.tail()

## 2. Train/Test Split
Given the small dataset (41 months), we use a small test set of the last 6 months to evaluate forecast performance.

In [ ]:
train_size = len(monthly_data) - 6
train = monthly_data.iloc[:train_size]
test = monthly_data.iloc[train_size:]

print(f"Train samples: {len(train)}")
print(f"Test samples: {len(test)}")

## 3. Model Implementation & Evaluation

We will define a dictionary to store predictions.

In [ ]:
predictions = pd.DataFrame(index=test.index)
predictions['Actual'] = test['attendance']

def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"--- {model_name} ---")
    print(f"RMSE: {rmse:,.0f}")
    print(f"MAE: {mae:,.0f}")
    print(f"MAPE: {mape:.2%}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

### 3.1 Seasonal Naive (Baseline)
Forecasts the value from the same month last year.

In [ ]:
# Shift by 12 months
predictions['S.Naive'] = monthly_data['attendance'].shift(12).iloc[train_size:]

# Handle edge case if shift produces NaNs (e.g. if test set is longer than history which is not here)
predictions['S.Naive'] = predictions['S.Naive'].fillna(method='bfill') # simple fallback

evaluate_model(predictions['Actual'], predictions['S.Naive'], "Seasonal Naive")

### 3.2 SARIMAX
Using (1,1,1)x(1,1,0,12) as a starting point based on seasonality.

In [ ]:
try:
    sarima_model = SARIMAX(train['attendance'], 
                           exog=train['covid_dummy'],
                           order=(1, 1, 1), 
                           seasonal_order=(1, 1, 0, 12),
                           enforce_stationarity=False,
                           enforce_invertibility=False)
    sarima_results = sarima_model.fit(disp=False)
    
    # Forecast
    sarima_pred = sarima_results.get_forecast(steps=len(test), exog=test[['covid_dummy']])
    predictions['SARIMAX'] = sarima_pred.predicted_mean
    
    evaluate_model(predictions['Actual'], predictions['SARIMAX'], "SARIMAX")
except Exception as e:
    print(f"SARIMAX failed: {e}")
    predictions['SARIMAX'] = np.nan

### 3.3 Exponential Smoothing (ETS / Holt-Winters)

In [ ]:
try:
    # 'add' for trend and seasonality is often good for attendance if amplitude doesn't scale with level
    # 'mul' if seasonal swings assume multiplicative growth. We'll try additive first data seems stable-ish level.
    ets_model = ExponentialSmoothing(train['attendance'], 
                                     trend='add', 
                                     seasonal='add', 
                                     seasonal_periods=12).fit()
    predictions['ETS'] = ets_model.forecast(len(test))
    
    evaluate_model(predictions['Actual'], predictions['ETS'], "ETS (Holt-Winters)")
except Exception as e:
    print(f"ETS failed: {e}")
    predictions['ETS'] = np.nan

### 3.4 Prophet

In [ ]:
# Prophet expects columns 'ds' and 'y'
prophet_train = train.reset_index()[['date', 'attendance']].rename(columns={'date': 'ds', 'attendance': 'y'})

m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
# Add covid as a regressor? Or let Prophet handle it? 
# Prophet handles outliers well, but COVID is a structural break.
# Let's keep it simple first without extra regressors to test robustness, 
# usually Prophet fits trend changes automatically if changepoints are allowed.

m.fit(prophet_train)

future = m.make_future_dataframe(periods=len(test), freq='MS')
forecast = m.predict(future)

predictions['Prophet'] = forecast.iloc[-len(test):]['yhat'].values
evaluate_model(predictions['Actual'], predictions['Prophet'], "Prophet")

### 3.5 Theta Method
Uses decomposition to forecast.

In [ ]:
try:
    theta_model = ThetaModel(train['attendance'], period=12)
    theta_res = theta_model.fit()
    predictions['Theta'] = theta_res.forecast(len(test))
    
    evaluate_model(predictions['Actual'], predictions['Theta'], "Theta")
except Exception as e:
    print(f"Theta Model failed: {e}")
    predictions['Theta'] = np.nan

### 3.6 Ensemble
Average of SARIMAX, ETS, Prophet, and Theta.

In [ ]:
cols_to_average = ['SARIMAX', 'ETS', 'Prophet', 'Theta']
cols_available = [c for c in cols_to_average if c in predictions.columns and not predictions[c].isna().any()]
predictions['Ensemble'] = predictions[cols_available].mean(axis=1)

evaluate_model(predictions['Actual'], predictions['Ensemble'], "Ensemble")

## 4. Final Comparison

In [ ]:
results = []
for col in predictions.columns:
    if col == 'Actual': continue
    
    try:
        r = evaluate_model(predictions['Actual'], predictions[col], col)
        r['Model'] = col
        results.append(r)
    except:
        pass

res_df = pd.DataFrame(results).sort_values('MAPE')

plt.figure(figsize=(12, 6))
sns.barplot(x='MAPE', y='Model', data=res_df, palette='viridis')
plt.title('Model Comparison (MAPE)')
plt.xlabel('MAPE')
plt.tight_layout()
plt.show()

print("Top Models by MAPE:")
display(res_df)

In [ ]:
# Plot forecasts
plt.figure(figsize=(15, 8))
plt.plot(train.index, train['attendance'], label='Train Data', color='black')
plt.plot(test.index, test['attendance'], label='Actual Test Data', color='black', linestyle='--', linewidth=2)

colors = ['blue', 'green', 'orange', 'purple', 'red']
for i, col in enumerate(['SARIMAX', 'ETS', 'Prophet', 'Theta', 'Ensemble']):
    if col in predictions.columns:
        plt.plot(predictions.index, predictions[col], label=col, alpha=0.7)

plt.title('Forecast Models Comparison')
plt.legend()
plt.show()